# Lesson 5.3: Confidence Intervals for a Proportion

Build 95% confidence intervals for a proportion with the Wald and Wilson methods, and see why Wilson is safer for small samples.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. A healthy poll: 476 of 850

The lesson's poll. statsmodels' proportion_confint takes the count of successes, the sample size, and a method: 'normal' is the classic Wald interval, 'wilson' is the Wilson score interval.

In [1]:
from statsmodels.stats.proportion import proportion_confint

x, n = 476, 850
print(f'p-hat = {x / n:.3f}')
for method in ('normal', 'wilson'):
    low, high = proportion_confint(x, n, alpha=0.05, method=method)
    print(f'{method:7s}: ({low:.4f}, {high:.4f})')

p-hat = 0.560
normal : (0.5266, 0.5934)
wilson : (0.5264, 0.5930)


## 2. A small sample: 3 of 20

Now only 3 of 20 patients had a complication. n times p-hat is just 3, below the rule of 10, so the Wald interval is not trustworthy here. The raw Wald formula gives a lower limit below 0 (about -0.006), which is impossible, so statsmodels cuts it off at 0.

In [2]:
x, n = 3, 20
print(f'p-hat = {x / n:.2f}')
for method in ('normal', 'wilson'):
    low, high = proportion_confint(x, n, alpha=0.05, method=method)
    print(f'{method:7s}: ({low:.3f}, {high:.3f})')

p-hat = 0.15
normal : (0.000, 0.306)
wilson : (0.052, 0.360)


## 3. No events at all: 0 of 20

With zero events, Wald collapses to (0, 0), which claims we are certain the rate is zero after only 20 patients. Wilson gives a sensible upper limit.

In [3]:
for method in ('normal', 'wilson'):
    low, high = proportion_confint(0, 20, alpha=0.05, method=method)
    print(f'{method:7s}: ({low:.3f}, {high:.3f})')

normal : (0.000, 0.000)
wilson : (0.000, 0.161)


## 4. Coverage test

Suppose the true rate is exactly 10% and n = 20. We simulate 20,000 samples, build both intervals each time and count how often they contain 10%. A good 95% interval should be close to 95%.

In [4]:
import numpy as np

rng = np.random.default_rng(3)
counts = rng.binomial(20, 0.10, size=20_000)
for method in ('normal', 'wilson'):
    low, high = proportion_confint(counts, 20, alpha=0.05, method=method)
    covered = (low <= 0.10) & (0.10 <= high)
    print(f'{method:7s}: covers the true 10% in {covered.mean():.1%} of samples')

normal : covers the true 10% in 87.5% of samples
wilson : covers the true 10% in 95.7% of samples


## Try this

- Change 3 of 20 to 6 of 40 (the same p-hat) and predict how the gap between Wald and Wilson changes.
- Try 1 of 20 and look at the Wald lower limit.
- In the coverage test, change the true rate to 0.5 and see whether Wald does better.

## The same idea in R, SPSS and Stata

**R**

```r
x <- 3; n <- 20; phat <- x / n
me <- 1.96 * sqrt(phat * (1 - phat) / n)
c(phat - me, phat + me)
prop.test(x, n, correct = FALSE)$conf.int
binom.test(x, n)$conf.int
prop.test(476, 850, correct = FALSE)$conf.int
```

**SPSS** (Transform > Compute Variable (SPSS menus offer exact and Jeffreys intervals under Analyze > Nonparametric Tests > One Sample, but not Wald or Wilson))

```
DATA LIST FREE / x n.
BEGIN DATA
3 20
END DATA.
COMPUTE phat = x / n.
COMPUTE z = IDF.NORMAL(0.975, 0, 1).
COMPUTE wlo = phat - z * SQRT(phat * (1 - phat) / n).
COMPUTE whi = phat + z * SQRT(phat * (1 - phat) / n).
COMPUTE ctr = (phat + z**2 / (2 * n)) / (1 + z**2 / n).
COMPUTE half = z * SQRT(phat * (1 - phat) / n + z**2 / (4 * n**2)) / (1 + z**2 / n).
COMPUTE slo = ctr - half.
COMPUTE shi = ctr + half.
LIST VARIABLES=wlo whi slo shi.
```

**Stata**

```
cii proportions 20 3, wald
cii proportions 20 3, wilson
cii proportions 20 3, exact
cii proportions 850 476, wald
cii proportions 850 476, wilson
```